In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from gate import IOStore
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
ios   = IOStore()
mdbpd = MusicDBPermDir()

In [ ]:
from lib import mymixtapez
mio   = mymixtapez.MusicDBIO(verbose=False)
webio = mymixtapez.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

In [ ]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [ ]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:        {0}".format(len(localArtists.get())))
print("   Master Artists:       {0}".format(len(masterArtists.get())))
print("   Master Artists Data:  {0}".format(len(masterArtistsData.get())))
print("   Errors:               {0}".format(len(errors.get())))
print("   Search Artists:       {0}".format(searchArtists.shape[0]))
print("   Known IDs:            {0}".format(len(knownArtists)))

# Download Search Data

In [ ]:
mio   = mymixtapez.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = mymixtapez.RawWebData(debug=False)

In [ ]:
useRYMData      = True
useSpotifyData  = False
useAllMusicData = False # Last one done
useLastFMData   = False
useAOTYData     = False

if useSpotifyData is True:
    spotio = ios.get(db="Spotify")
    spotGenreData = DataFrame(spotio.data.getSummaryNameData()).join(spotio.data.getSummaryGenreData())
    def isRap(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    spotGenreData = spotGenreData[spotGenreData["Genre"].notna()]
    artistNames = spotGenreData[spotGenreData["Genre"].apply(isRap)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useAllMusicData is True:
    from gate import MusicDBGate
    mdbgate = MusicDBGate()
    amio = mdbgate.getIO(db="AllMusic")
    amGenreData = DataFrame(amio.data.getSummaryNameData()).join(amio.data.getSummaryGenreData())
    def isRap(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    amGenreData = amGenreData[amGenreData["Tag"].notna()]
    artistNames = amGenreData[amGenreData["Tag"].apply(isRap)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useRYMData is True:
    rymio = ios.get(db="RateYourMusic")
    rymGenreData = DataFrame(rymio.data.getSummaryNameData()).join(rymio.data.getSummaryGenreData())
    def isRap(genres):
        if isinstance(genres,list):
            tests = ['Trap', 'Southern Hip Hop', 'Cloud Rap', 'Tread', 'Pop Rap', 'Vaportrap', 'East Coast Hip Hop', 'Contemporary R&B', 'Gangsta Rap', 'UK Drill', 'Experimental Hip Hop', 'Alternative R&B', 'Plugg', 'Hip House', 'Conscious Hip Hop', 'Pop Punk', 'Future House', 'Instrumental Hip Hop', 'Tech House', 'Neo-Soul', 'Jook', 'Film Soundtrack', 'Synthpop', 'Disco Rap', 'Hardcore Hip Hop', 'Dirty South', 'Chipmunk Soul', 'Hip Hop', 'UK Hip Hop', 'West Coast Hip Hop', 'Comedy Rap', 'Industrial Hip Hop', 'Memphis Rap', 'Trap [EDM]', 'Dance-Pop', 'Electropop', 'Pop', 'Electro House', 'Teen Pop', 'Tropical House', 'Jerk Rap', 'Chopped and Screwed', 'Mobb Music', 'Hyphy', 'Political Hip Hop', 'Boom Bap', 'Drill', 'Rap Rock', 'Novelty', 'Video Game Music', 'Emo Rap', 'Afrobeats', 'Dancehall', 'Trap Metal', 'Hyperpop', 'Neoperreo', 'Jazz Rap', 'Ratchet Music', 'Future Bass', 'Electro', 'Freestyle', 'G-Funk', 'Grime', '2-Step', 'UK Garage', 'Pop Rock', 'Indie Pop', 'Christmas Music', 'Abstract Hip Hop', 'Mashup', 'Pop Reggae', 'Country Pop', 'Pop Soul', 'Crunk', 'Soul', 'Singer/Songwriter', 'Detroit Trap', 'Snap', 'Alternative Rock', 'Contemporary Country', 'Country Rap', 'Country Rock', 'Art Pop', 'Folk Pop', 'Drumless', 'Liquid Drum and Bass', 'Ghettotech', 'Miami Bass', 'Digicore', 'Rage', 'Reggaeton', 'Latin Pop', 'Latin Rap', 'Hard Trap', 'Slap House', 'Digital Hardcore', 'Wonky', 'Bop', 'Sounds and Effects', 'Television Music', 'Educational Music', 'A cappella', 'Lo-Fi Hip Hop', 'Smooth Soul', 'Funk', 'Synth Funk', 'Horrorcore', 'Futuristic Swag', 'Afroswing', 'Christian Hip Hop', 'New Age', 'UK Bass', 'Ballroom', 'Emo-Pop', 'Bounce', 'Spoken Word', 'Chicano Rap', 'Classical Crossover']
            for test in tests:
                for genre in genres:
                    if genre.find(test) != -1:
                        return True
            return False

    rymGenreData = rymGenreData[rymGenreData["Genre"].notna()]
    artistNames = rymGenreData[rymGenreData["Genre"].apply(isRap)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("# {0} Search Results".format(db))
    print("#   Available Names:      {0}".format(len(artistNames)))
    print("#   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  8820
#   Artist Names To Get:  7209

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "10:50am")
tt = TermTime("today", "8:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    #if searchedForErrors.get(artistName) is not None:
    #    continue

    response = webio.getArtistSearchData(artistName=artistName)
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(5.0)
        continue
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(5.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
    masterArtists.save(data=masterArtistsDict)
    masterArtistsData.save(data=masterArtistsDataDict)
    errors.save(data=searchedForErrors)

In [ ]:
refs = []
for name,result in masterArtistsData.get().items():
    for artistID,artistName in result.items():
        refs.append({"ID": artistID, "Name": artistName})
df = DataFrame(refs)
df.index = df["ID"]
df = df[["Name"]]
N  = df.index.value_counts()
N.name = "Num"
df = df.drop_duplicates()
df.index.name = None
df = df.join(N)
df = df.sort_values(by='Num', ascending=False)
searchArtistsData = mymixtapez.MusicDBIO(local=False).data.getSearchArtistData()
print("Found {0} Previous Search Artists".format(searchArtistsData.shape[0]))
print("Found {0} New Search Artists".format(df.shape[0]))
searchArtistsData = concat([searchArtistsData,df]).drop_duplicates()
print("Found {0} Unique Search Artists".format(searchArtistsData.shape[0]))
mymixtapez.MusicDBIO(local=False).data.saveSearchArtistData(data=searchArtistsData)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = mymixtapez.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = mymixtapez.RawWebData(debug=False)

In [ ]:
useSearchData  = True
useStarterData = False

if useStarterData is True:
    artistNames      = io.get("starter.p")
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useSearchData is True:
    artistNames      = searchArtists.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())]["Name"]

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  37325
#   Artist Names To Get:  28198
#   Artist Names To Get:  18771
#   Artist Names To Get:  9874

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "10:50am")
tt = TermTime("today", "8:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,artistName) in enumerate(artistNamesToGet.iteritems()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(6.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:

localArtists.save(data=localArtistsDict)

In [ ]:
from lib.mymixtapez import moveLocalFiles
moveLocalFiles()

In [ ]:
mdbio = mymixtapez.MusicDBIO(local=False)

In [ ]:
mdbio.prd.parse(0, verbose=True, force=True)

In [ ]:
from utils import PoolIO
pio = PoolIO("MyMixTapez")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

# Create Starter

In [ ]:
from ioutils import HTMLIO, FileIO
import json
hio = HTMLIO()
io = FileIO()
artists = {}
bsdata = hio.get(io.get("../../sandbox/recent_mymixtapez.p"))
jTag = bsdata.find("script", {"type": "application/json"})
jData = json.loads(jTag.text.replace('&q;', '"'))
for item in jData["album-recents-page"]['success']['recents']:
    artists.update({artist['id']: artist['name'] for artist in item['artists']['main']})
    
bsdata = hio.get(io.get("../../sandbox/trending_mymixtapez.p"))
jTag = bsdata.find("script", {"type": "application/json"})
jData = json.loads(jTag.text.replace('&q;', '"'))
for item in jData["trending-artists-page"]['success']['items']:
    artists.update({item['id']: item['name']})

In [ ]:
io.save(idata=Series(artists), ifile="starter.p")

In [ ]:
from collections import Counter
import json
from ioutils import FileIO, HTMLIO
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
mio = mymixtapez.MusicDBIO(local=False)
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p", debug=False):
        bsdata = hio.get(io.get(ifile))
        jTag = bsdata.find("script", {"type": "application/json"})
        jData = json.loads(jTag.text.replace('&q;', '"'))
        for item in jData['artist-page']['success']['alsoAppears']:
            for artist in item['artists']['main']:
                refs.append(artist)
    if (modVal+1) % 25 == 0:
        print(modVal+1,'\t',len(refs))

In [ ]:
df = DataFrame(refs)
N  = df['id'].value_counts()
N.name = "Num"
df = df.drop_duplicates()
df.index = df['id']
df.index.name = None
df = df.drop(['id'], axis=1)
df.columns = ["Name"]
df = df.join(N)
df = df.sort_values(by='Num', ascending=False)
print(df.shape)
mio = mymixtapez.MusicDBIO(local=False)
mio.data.saveSearchArtistData(data=df)